<a href="https://colab.research.google.com/github/MahathiKadiyala/24B11DS084_DriveReady_OPC_2026/blob/main/Day-2/EV_Service_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

!pip install -q openai


In [4]:

import os
import json
from openai import OpenAI
from google.colab import userdata

groq_api_key = userdata.get("agentic_ai_GROQ_API_KEY")

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

print("EV Service Assistant API setup completed!")


EV Service Assistant API setup completed!


In [5]:

EV_SERVICES = {
    "battery_health_check": 500,
    "general_service": 800,
    "brake_inspection": 300,
    "motor_diagnosis": 1000,
    "software_update": 400,
    "charging_system_check": 600
}

print("Available EV Services:\n")

for service, price in EV_SERVICES.items():
    print(f"{service.replace('_', ' ').title()} : Rs {price}")


Available EV Services:

Battery Health Check : Rs 500
General Service : Rs 800
Brake Inspection : Rs 300
Motor Diagnosis : Rs 1000
Software Update : Rs 400
Charging System Check : Rs 600


In [7]:

EV_SYSTEM_PROMPT = """
You are an intelligent EV Service Assistant.

Your job is to help electric vehicle owners with:

1. Battery health checks
2. General EV servicing
3. Brake inspection
4. Motor diagnosis
5. Software updates
6. Charging system checks

Sample service prices:
- Battery health check: Rs 500
- General service: Rs 800
- Brake inspection: Rs 300
- Motor diagnosis: Rs 1000
- Software update: Rs 400
- Charging system check: Rs 600

Rules:
- Greet customers politely.
- Ask for vehicle details when needed.
- Explain services in simple language.
- Provide prices only from the given service list.
- If a service is unavailable, say so.
- Never invent a vehicle diagnosis.
- Do not claim that a service appointment is booked.
- Ask the customer to contact a service centre for
  actual appointments and technical inspections.
- Never request passwords, OTPs or payment details.
"""

print("System prompt created!")


System prompt created!


In [8]:

def ask_llm(messages):

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.3
    )

    return response.choices[0].message.content


print("LLM function created!")


LLM function created!


In [9]:

def ev_chat(user_message):

    messages = [
        {"role": "system", "content": EV_SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

    return ask_llm(messages)


print(ev_chat("Hello, I need help with my electric scooter."))


Hello! 👋  
I’m here to help you with your electric scooter. Could you let me know what specific issue or service you’re looking for? For example, are you concerned about battery health, brakes, motor performance, software updates, or the charging system? Once I know a bit more, I can explain the available options and their prices.


In [10]:
print(ev_chat("What services do you provide for EVs?"))

Hello! 👋  
I’m here to help you with a range of electric‑vehicle services. Here’s what we offer:

| Service | Description | Price |
|---------|-------------|-------|
| **Battery health check** | Quick scan of your battery’s condition and performance. | ₹500 |
| **General EV servicing** | Routine maintenance—fluid checks, filter changes, and a quick system check. | ₹800 |
| **Brake inspection** | Safety check of brake pads, discs, and hydraulic system. | ₹300 |
| **Motor diagnosis** | Diagnostic test of the electric motor and related electronics. | ₹1,000 |
| **Software update** | Latest firmware and software upgrades for your vehicle’s control systems. | ₹400 |
| **Charging system check** | Inspection of on‑board charger, charging port, and related wiring. | ₹600 |

If you’d like to schedule a service or need a detailed inspection, please contact your local service centre. They’ll set up an appointment and handle the technical work. Let me know if you have any questions or need more de

In [11]:

CLASSIFICATION_PROMPT = """
You are an EV service request classifier.

Classify the customer's message into exactly one category:

battery
general_service
brakes
motor
software_update
charging
other

Examples:

Customer: My battery is draining quickly.
Category: battery

Customer: I want to service my electric scooter.
Category: general_service

Customer: My brakes are making noise.
Category: brakes

Customer: My motor is making strange sounds.
Category: motor

Customer: I need the latest software update.
Category: software_update

Customer: My charger is not working.
Category: charging

Return only the category name.
"""


def classify_ev_request(message):

    messages = [
        {"role": "system", "content": CLASSIFICATION_PROMPT},
        {"role": "user", "content": message}
    ]

    return ask_llm(messages).strip().lower()


print(classify_ev_request("My EV battery is draining very fast."))
print(classify_ev_request("I want to inspect my brakes."))
print(classify_ev_request("My charger is not working."))


battery
brakes
charging


In [12]:

def extract_ev_details(message):

    prompt = """
    Extract the following information from the customer message:

    {
        "customer_name": "",
        "vehicle_type": "",
        "vehicle_model": "",
        "service_required": [],
        "problem_description": ""
    }

    Rules:
    - Return valid JSON only.
    - Use empty strings for missing text fields.
    - Use an empty list if services are not mentioned.
    - Do not invent missing information.
    """

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": message}
    ]

    result = ask_llm(messages)

    # Remove Markdown code fences if returned
    result = result.strip()

    if result.startswith("```"):
        result = result.replace("```json", "", 1)
        result = result.replace("```", "")
        result = result.strip()

    return json.loads(result)


customer_message = """
My name is Ravi.
I own an Ola S1 Pro electric scooter.
I need a battery health check and brake inspection.
My battery is draining quickly.
"""

customer_details = extract_ev_details(customer_message)

print(json.dumps(customer_details, indent=4))


{
    "customer_name": "Ravi",
    "vehicle_type": "electric scooter",
    "vehicle_model": "Ola S1 Pro",
    "service_required": [
        "battery health check",
        "brake inspection"
    ],
    "problem_description": "My battery is draining quickly."
}


In [13]:

def calculate_service_cost(services):

    total = 0

    for service in services:

        if service not in EV_SERVICES:
            raise ValueError(f"Unknown service: {service}")

        total += EV_SERVICES[service]

    return total


selected_services = [
    "battery_health_check",
    "brake_inspection"
]

total_cost = calculate_service_cost(selected_services)

print("Selected Services:")

for service in selected_services:
    print(service.replace("_", " ").title())

print("Total Estimated Cost: Rs", total_cost)


Selected Services:
Battery Health Check
Brake Inspection
Total Estimated Cost: Rs 800


In [14]:

def log_execution(func):

    def wrapper(*args, **kwargs):

        print(f"Executing: {func.__name__}")

        result = func(*args, **kwargs)

        print(f"Completed: {func.__name__}")

        return result

    return wrapper


@log_execution
def calculate_ev_bill(services):

    return calculate_service_cost(services)


bill = calculate_ev_bill([
    "battery_health_check",
    "motor_diagnosis"
])

print("Estimated Bill: Rs", bill)


Executing: calculate_ev_bill
Completed: calculate_ev_bill
Estimated Bill: Rs 1500


In [17]:
from openai import RateLimitError, AuthenticationError, APIConnectionError

def safe_ev_chat(message):

    try:
        return ev_chat(message)

    except RateLimitError:
        return "Server is busy. Please try again later."

    except AuthenticationError:
        return "API key is invalid. Please check your setup."

    except APIConnectionError:
        return "Connection problem. Check your internet."

    except Exception as error:
        return f"An unexpected error occurred: {error}"

In [18]:

ev_history = []

MAX_TURNS = 5


def ev_chat_with_memory(user_message):

    ev_history.append({
        "role": "user",
        "content": user_message
    })

    # Keep only the latest 5 user-assistant exchanges
    recent_history = ev_history[-(MAX_TURNS * 2):]

    messages = [
        {"role": "system", "content": EV_SYSTEM_PROMPT}
    ] + recent_history

    try:
        answer = ask_llm(messages)

        ev_history.append({
            "role": "assistant",
            "content": answer
        })

        return answer

    except Exception:
        ev_history.pop()
        raise


print(ev_chat_with_memory("My name is Ravi."))
print(ev_chat_with_memory("I own an Ola S1 Pro."))
print(ev_chat_with_memory("What vehicle do I own?"))


Hello Ravi! 👋 How can I assist you with your electric vehicle today? If you’re looking into a service or have a question about your car, I’ll need a few details first—such as the make, model, and year of your vehicle. Once I have that, I can explain the available services and their prices.
Great, thanks for sharing!  
Could you let me know the year of your Ola S1 Pro? That will help me give you the most accurate information about the services that are best suited for your vehicle.
You mentioned that you own an **Ola S1 Pro**. If you need any service information or have questions about that vehicle, just let me know!


In [19]:

test_messages = [
    "Hello, my name is Ravi.",
    "I own an Ola S1 Pro electric scooter.",
    "My battery is draining quickly.",
    "What services can help me?",
    "How much does a battery health check cost?",
    "What vehicle do I own?"
]

for message in test_messages:

    print("\nYou:", message)

    response = safe_ev_chat(message)

    print("EV Assistant:", response)



You: Hello, my name is Ravi.
EV Assistant: Hello Ravi! 👋  
How can I help you with your electric vehicle today? If you’re looking for a specific service—like a battery health check, brake inspection, or a software update—just let me know, and I’ll give you the details and the price. If you need more information about your vehicle or have any questions, feel free to share!

You: I own an Ola S1 Pro electric scooter.
EV Assistant: Hello! 👋  
Thanks for reaching out. How can I help you with your Ola S1 Pro today?  
If you’re looking for a specific service—like a battery health check, brake inspection, or a software update—just let me know, and I’ll give you the details and the price.  

Feel free to share any symptoms or concerns you’ve noticed, and I’ll guide you on the next steps.

You: My battery is draining quickly.
EV Assistant: Hello! I’m here to help with your EV battery issue.

It sounds like a **battery health check** would be a good first step. This test will let us see how the

In [20]:

print("Welcome to EV Service Assistant!")
print("Ask me about EV services, batteries, brakes and more.")
print("Type 'bye' to exit.\n")

while True:

    user_input = input("You: ")

    if user_input.lower().strip() == "bye":
        print("EV Assistant: Thank you for using our service. Goodbye!")
        break

    if not user_input.strip():
        print("EV Assistant: Please enter a message.")
        continue

    response = safe_ev_chat(user_input)

    print("EV Assistant:", response)


Welcome to EV Service Assistant!
Ask me about EV services, batteries, brakes and more.
Type 'bye' to exit.

You: bye
EV Assistant: Thank you for using our service. Goodbye!
